# 18 — Week 5: hybrid QNN on validation: design screen, three seeds, controls, Diebold–Mariano

Needs notebook 17 (backend decision, `reports/week5/qnn_smoke.json`, seed-0 runs in `experiments/week5_qnn/`).
Same protocol as every network in the benchmark (Adam 1e-3, batch 512, patience 3, epoch cap 30, grad clip 1, local
scaling, univariate 60-minute window, masked MSE on h = 1/5/15).

1. **Design screen (seed 0, validation only):** 4 qubits, layers {2, 4} x data re-uploading {on, off}. The variant with
   the lowest mean validation MAE becomes the main hybrid QNN (provisional). Qubit count and data fraction are Week 6.
2. **Three seeds** of the main hybrid QNN, its entanglement ablation, the parameter-matched MLP control and the
   bottleneck control.
3. Table (mean ± sd over seeds) merged with the frozen Week 4 validation benchmark.
4. Diebold–Mariano with Holm correction (seed 0) against the controls and the classical references (PatchTST,
   `gru_uni`, persistence), and ramp breakdown.

Cell 3 repeats the overfit sanity check in its textbook form (memorise 64 samples), because the 1% check of notebook 17
used a loss-reduction threshold that was too strict for noisy one-minute changes (all training skills were positive).
Every run is appended to `reports/week5/quantum_variants_log.csv`. **The test split is not used.**

In [1]:
# Cell 1: setup (reads the decisions of notebook 17)
import sys, json, time, math, shutil, warnings, traceback
from pathlib import Path
import numpy as np
import polars as pl
import torch

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
from models import rnn as R
from models import quantum as Q
from evaluation import compare as C

GROUP = "week5_qnn"
REP = ROOT / "reports" / "week5"
SMOKE = REP / "qnn_smoke.json"
assert SMOKE.exists(), "run notebook 17 first (reports/week5/qnn_smoke.json missing)"
smoke = json.loads(SMOKE.read_text())
BACKEND = smoke["timing"]["backend"]
MAX_TRAIN = smoke["timing"]["max_train_samples"]
PROTOCOL = {**smoke["protocol"], "backend": BACKEND}
EPOCH_CAP = int(PROTOCOL["epochs"])
first = pl.read_csv(REP / "qnn_first_val_s0.csv")
cap = first.filter(pl.col("best_epoch") == EPOCH_CAP)["model"].to_list()
if any(m.startswith("hqnn") for m in cap):
    # Provisional rule: the small Week 5 models are cheap; if the hybrid QNN of notebook 17 stopped at the cap, every
    # model of this notebook (hybrid and controls) gets 60 epochs, in a separate group so nothing is mixed.
    EPOCH_CAP = 60
    PROTOCOL["epochs"] = EPOCH_CAP
    GROUP = "week5_qnn_e60"
DATA_CFG = {"scaling": "local", "calendar": False, "covariates": False}
SEEDS = [0, 1, 2]
SUMMARY = {"backend": BACKEND, "max_train_samples": MAX_TRAIN}
SUMMARY.update({"epoch_cap": EPOCH_CAP, "group": GROUP})
print(f"backend {BACKEND} | max_train_samples {MAX_TRAIN} | epoch cap {EPOCH_CAP} | unit checks of nb 17: "
      f"{all(smoke['unit_checks'].values())}")
print(first.filter(~pl.col("model").str.starts_with("[ref]")).select("model", "skill_h1", "skill_h5", "skill_h15",
                                                                     "best_epoch"))
print("runs of notebook 17 whose best epoch equals the cap of 30:", cap or "none", "| epoch cap here:", EPOCH_CAP,
      "| group:", GROUP)

backend default.qubit | max_train_samples None | epoch cap 30 | unit checks of nb 17: True
shape: (4, 5)
┌───────────────────────┬──────────┬──────────┬───────────┬────────────┐
│ model                 ┆ skill_h1 ┆ skill_h5 ┆ skill_h15 ┆ best_epoch │
│ ---                   ┆ ---      ┆ ---      ┆ ---       ┆ ---        │
│ str                   ┆ f64      ┆ f64      ┆ f64       ┆ i64        │
╞═══════════════════════╪══════════╪══════════╪═══════════╪════════════╡
│ hqnn_q4_l2            ┆ 0.0421   ┆ 0.0512   ┆ 0.0377    ┆ 10         │
│ ctrl_mlp_q4_l2        ┆ 0.0359   ┆ 0.0554   ┆ 0.0434    ┆ 15         │
│ ctrl_bottleneck_q4_l2 ┆ 0.083    ┆ 0.0439   ┆ 0.0305    ┆ 5          │
│ hqnn_noent_q4_l2      ┆ 0.0784   ┆ 0.0526   ┆ 0.041     ┆ 15         │
└───────────────────────┴──────────┴──────────┴───────────┴────────────┘
runs of notebook 17 whose best epoch equals the cap of 30: none | epoch cap here: 30 | group: week5_qnn


In [2]:
# Cell 2: DRY RUN on synthetic data for the variants of this notebook (2 epochs; deleted afterwards)

def synth(n=30000, seed=0):
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    x = np.zeros(n)
    e = rng.normal(0, 0.4, n)
    for i in range(1, n):
        x[i] = 0.9 * x[i - 1] + e[i]
    P = 30 + np.cumsum(rng.normal(0, 0.01, n)) + x - 0.3 * (t % 5 == 0)
    valid = np.ones(n, bool)
    for g0 in rng.choice(np.arange(500, n - 500), 6, replace=False):
        valid[g0:g0 + 150] = False
    i_tr, i_va = int(0.7 * n), int(0.85 * n)
    split = np.where(t < i_tr, "train", np.where(t < i_va, "val", "test"))
    c = np.concatenate([[0], np.cumsum(~valid)])
    win_ok = np.zeros(n, bool)
    win_ok[R.INPUT_LEN - 1:] = (c[R.INPUT_LEN:] - c[:n - R.INPUT_LEN + 1]) == 0
    Pv = np.where(valid, P, np.nan)
    L24 = pl.Series(Pv).fill_nan(None).rolling_mean(1440, min_samples=720).to_numpy()
    cols = {"minute_epoch": t * 60, "P_kW": Pv, "valid": valid, "split": split, "mod5": (t % 5).astype(np.int64),
            "hour_sin": np.sin(2 * np.pi * (t // 60 % 24) / 24), "hour_cos": np.cos(2 * np.pi * (t // 60 % 24) / 24),
            "L24_kW": L24}
    thr = {}
    for h in R.HORIZONS:
        ok = win_ok.copy()
        ok[n - h:] = False
        ok[:n - h] &= valid[h:]
        sp = np.full(n, None, dtype=object)
        tt = np.arange(n - h)
        sp[:n - h] = np.where(tt + h < i_tr, "train", np.where((tt >= i_tr) & (tt + h < i_va), "val",
                              np.where(tt >= i_va, "test", None)))
        ok &= sp != None  # noqa: E711
        rel = np.full(n, np.nan)
        rel[:n - h] = np.abs(P[h:] - P[:n - h]) / L24[:n - h]
        trm = ok & (sp == "train") & np.isfinite(rel)
        thr[str(h)] = {"rel_p95": float(np.quantile(rel[trm], 0.95))}
        ramp = np.where(ok & np.isfinite(rel), (rel > thr[str(h)]["rel_p95"]).astype(float), np.nan)
        cols[f"ok_h{h}"], cols[f"split_h{h}"], cols[f"ramp_h{h}"] = ok, sp.tolist(), ramp
    df = pl.DataFrame(cols).with_columns(pl.col("P_kW").fill_nan(None), pl.col("L24_kW").fill_nan(None))
    return df, {"thresholds": thr}


DRY = "_dryrun_week5b"
sm, smeta = synth()
sdata = R.Data(sm, DATA_CFG)
dry_ok = True
try:
    for v in [{"n_layers": 4, "reupload": True}, {"n_layers": 2, "reupload": False}, {"n_layers": 4, "middle": "mlp"}]:
        cfg = {**PROTOCOL, "n_qubits": 4, "middle": "quantum", "entangle": True, **v, "seed": 0, "epochs": 2,
               "max_train_samples": 3000}
        res = Q.run_and_score(ROOT, sm, smeta, sdata, cfg, DRY, tracking_mode="disabled", log=lambda s: None)
        finite = all(np.isfinite(r_["MAE"]) for r_ in res["val"])
        dry_ok &= bool(finite)
        print(f"{Q.variant_name(cfg):22s} params {res['info']['n_params']} (middle {res['info']['n_params_middle']}) | "
              f"finite {finite}")
except Exception:
    dry_ok = False
    traceback.print_exc()
finally:
    shutil.rmtree(ROOT / "experiments" / DRY, ignore_errors=True)
    shutil.rmtree(ROOT / "experiments" / "runs" / DRY, ignore_errors=True)
    log_p = REP / "quantum_variants_log.csv"
    if log_p.exists():
        lg = pl.read_csv(log_p, infer_schema_length=0)
        lg = lg.filter(pl.col("group") != DRY) if "group" in lg.columns else lg
        lg.write_csv(log_p)
print("DRY RUN PASSED" if dry_ok else "DRY RUN FAILED: send this output")

hqnn_q4_l4             params 311 (middle 48) | finite True
hqnn_noreup_q4_l2      params 287 (middle 24) | finite True
ctrl_mlp_q4_l4         params 312 (middle 49) | finite True
DRY RUN PASSED


In [3]:
# Cell 3: OVERFIT SANITY CHECK v2 (memorisation). The 1% check of notebook 17 (1,967 noisy samples, 60 epochs) gave
# positive training skill for every model, but the loss fell only 9-18%, below the 20% threshold chosen there (a
# threshold that was too strict for mostly unpredictable 1-minute changes). The textbook form of the check is to
# memorise a tiny set: 64 train origins, full batch, lr 1e-2, 500 epochs, no early stopping. PASS = final training loss
# <= 30% of the first-epoch loss (hybrid QNN and MLP control; the bottleneck control is reported only).
assert dry_ok, "run cell 2 first"
md = pl.read_parquet(ROOT / PROTOCOL["dataset"])
meta = json.loads((ROOT / PROTOCOL["dataset_meta"]).read_text())
data = R.Data(md, DATA_CFG)
tiny = Q.subsample(data.origins("train"), 64, seed=0)
mem = []
for mid in ("quantum", "mlp", "bottleneck"):
    cfg = {**PROTOCOL, "n_qubits": 4, "n_layers": 2, "reupload": True, "entangle": True, "middle": mid, "seed": 0,
           "lr": 1e-2, "batch": 64, "epochs": 500, "patience": 500, "grad_clip": 0, "val_split": "train"}
    t0 = time.time()
    m_, d_, hist, info = Q.train(md, cfg, log=lambda s: None, data=data, train_origins=tiny, val_origins=tiny)
    l0, l1 = hist[0]["train_loss"], hist[-1]["train_loss"]
    ok = (l1 <= 0.3 * l0) if mid != "bottleneck" else None
    mem.append({"variant": info["variant"], "n_params": info["n_params"], "n_samples": len(tiny),
                "loss_epoch1": round(l0, 4), "loss_last": round(l1, 4), "ratio": round(l1 / l0, 3),
                "s": round(time.time() - t0, 1), "check": "PASS" if ok else ("report only" if ok is None else "FAIL")})
    Q.log_variant(ROOT, {"stage": "memorise_64", "group": GROUP, "variant": info["variant"], "n_params": info["n_params"],
                         "loss_ratio": round(l1 / l0, 3)})
mem = pl.DataFrame(mem)
print(mem)
MEM_OK = all(c == "PASS" for c in mem.filter(pl.col("check") != "report only")["check"])
SUMMARY["memorise_64"] = {"table": mem.to_dicts(), "passed": MEM_OK}
print("MEMORISATION CHECK PASSED (the training pipeline can fit data)" if MEM_OK else "MEMORISATION CHECK FAILED: send this output")

shape: (3, 8)
┌─────────────────────┬──────────┬───────────┬─────────────┬───────────┬───────┬─────┬─────────────┐
│ variant             ┆ n_params ┆ n_samples ┆ loss_epoch1 ┆ loss_last ┆ ratio ┆ s   ┆ check       │
│ ---                 ┆ ---      ┆ ---       ┆ ---         ┆ ---       ┆ ---   ┆ --- ┆ ---         │
│ str                 ┆ i64      ┆ i64       ┆ f64         ┆ f64       ┆ f64   ┆ f64 ┆ str         │
╞═════════════════════╪══════════╪═══════════╪═════════════╪═══════════╪═══════╪═════╪═════════════╡
│ hqnn_q4_l2          ┆ 287      ┆ 64        ┆ 2.2506      ┆ 0.4681    ┆ 0.208 ┆ 5.9 ┆ PASS        │
│ ctrl_mlp_q4_l2      ┆ 285      ┆ 64        ┆ 2.0025      ┆ 0.244     ┆ 0.122 ┆ 0.9 ┆ PASS        │
│ ctrl_bottleneck_q4_ ┆ 263      ┆ 64        ┆ 2.2155      ┆ 0.2075    ┆ 0.094 ┆ 0.9 ┆ report only │
│ l2                  ┆          ┆           ┆             ┆           ┆       ┆     ┆             │
└─────────────────────┴──────────┴───────────┴─────────────┴───────────┴─────

In [4]:
# Cell 4: DESIGN SCREEN, seed 0, validation (4 qubits; layers x re-uploading). Resumable; about 10-30 min.
SCREEN = [{"n_layers": 2, "reupload": True}, {"n_layers": 4, "reupload": True},
          {"n_layers": 2, "reupload": False}, {"n_layers": 4, "reupload": False}]
scr = []
for v in SCREEN:
    cfg = {**PROTOCOL, "n_qubits": 4, "middle": "quantum", "entangle": True, **v, "seed": 0}
    if MAX_TRAIN:
        cfg["max_train_samples"] = MAX_TRAIN
    res = Q.run_and_score(ROOT, md, meta, data, cfg, GROUP)
    scr.append({"variant": Q.variant_name(cfg), **v, "n_params": res["info"]["n_params"],
                "n_params_middle": res["info"]["n_params_middle"],
                **{f"val_MAE_h{r_['h']}": r_["MAE"] for r_ in res["val"]},
                **{f"skill_h{r_['h']}": r_["skill_MAE"] for r_ in res["val"]},
                "val_MAE_mean": float(np.mean([r_["MAE"] for r_ in res["val"]])),
                "best_epoch": res["info"]["best_epoch"]})
scr = pl.DataFrame(scr).sort("val_MAE_mean")
print(scr.select("variant", "n_params", "n_params_middle", pl.col("val_MAE_mean").round(5),
                 *[pl.col(f"skill_h{h}").round(4) for h in R.HORIZONS], "best_epoch"))
scr.write_csv(REP / "qnn_design_screen_s0.csv")
best = scr.row(0, named=True)
MAIN = {"n_qubits": 4, "n_layers": int(best["n_layers"]), "reupload": bool(best["reupload"]), "entangle": True}
gap = float(scr["val_MAE_mean"][1] - scr["val_MAE_mean"][0])
SUMMARY["design_screen"] = scr.to_dicts()
SUMMARY["main_hybrid"] = MAIN
print(f"\nMAIN hybrid QNN (provisional, chosen on validation): {Q.variant_name(MAIN)} | margin to the second variant "
      f"{gap:.5f} kW (mean MAE); a margin this small is within seed noise if it is below ~0.002 kW")
print("variants at the epoch cap:", scr.filter(pl.col("best_epoch") == EPOCH_CAP)["variant"].to_list() or "none")

[15:54:50] hqnn_q4_l2_s0: done earlier, skipped
[15:54:50] hqnn_q4_l4_s0: training


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/aryaneuse/.netrc.


[15:57:57] hqnn_q4_l4_s0: params 311 (middle 48) | train 167 s | best epoch 16/19 | val MAE 0.2756 / 0.4062 / 0.5778 kW | skill +0.038 / +0.055 / +0.045
[15:57:57] hqnn_noreup_q4_l2_s0: training
[15:59:43] hqnn_noreup_q4_l2_s0: params 287 (middle 24) | train 97 s | best epoch 19/22 | val MAE 0.2697 / 0.4056 / 0.5770 kW | skill +0.059 / +0.057 / +0.046
[15:59:43] hqnn_noreup_q4_l4_s0: training
[16:00:44] hqnn_noreup_q4_l4_s0: params 311 (middle 48) | train 48 s | best epoch 4/7 | val MAE 0.2734 / 0.4086 / 0.5827 kW | skill +0.046 / +0.050 / +0.037
shape: (4, 8)
┌─────────────┬──────────┬─────────────┬────────────┬──────────┬──────────┬───────────┬────────────┐
│ variant     ┆ n_params ┆ n_params_mi ┆ val_MAE_me ┆ skill_h1 ┆ skill_h5 ┆ skill_h15 ┆ best_epoch │
│ ---         ┆ ---      ┆ ddle        ┆ an         ┆ ---      ┆ ---      ┆ ---       ┆ ---        │
│ str         ┆ i64      ┆ ---         ┆ ---        ┆ f64      ┆ f64      ┆ f64       ┆ i64        │
│             ┆          ┆ i6

In [5]:
# Cell 5: THREE SEEDS: main hybrid QNN, entanglement ablation, MLP control (matched to the main circuit), bottleneck.
# Resumable (seed-0 runs of the screen / notebook 17 are reused). About 20-60 min.
FAMILY = [{"middle": "quantum"}, {"middle": "quantum", "entangle": False}, {"middle": "mlp"}, {"middle": "bottleneck"}]
T = time.time()
for seed in SEEDS:
    for v in FAMILY:
        cfg = {**PROTOCOL, **MAIN, **v, "seed": seed}
        if MAX_TRAIN:
            cfg["max_train_samples"] = MAX_TRAIN
        try:
            Q.run_and_score(ROOT, md, meta, data, cfg, GROUP)
        except Exception:
            print(f"!!! {Q.variant_name(cfg)}_s{seed} failed:")
            traceback.print_exc(limit=5)
print(f"\nall runs handled in {(time.time() - T) / 60:.1f} min")
NAMES = [Q.variant_name({**MAIN, **v}) for v in FAMILY]
print("models:", NAMES)

[16:00:44] hqnn_noreup_q4_l2_s0: done earlier, skipped
[16:00:44] hqnn_noent_noreup_q4_l2_s0: training
[16:02:08] hqnn_noent_noreup_q4_l2_s0: params 287 (middle 24) | train 67 s | best epoch 15/18 | val MAE 0.2639 / 0.4053 / 0.5783 kW | skill +0.079 / +0.057 / +0.044
[16:02:08] ctrl_mlp_q4_l2_s0: done earlier, skipped
[16:02:08] ctrl_bottleneck_q4_l2_s0: done earlier, skipped
[16:02:08] hqnn_noreup_q4_l2_s1: training
[16:03:15] hqnn_noreup_q4_l2_s1: params 287 (middle 24) | train 53 s | best epoch 10/13 | val MAE 0.2626 / 0.4060 / 0.5792 kW | skill +0.084 / +0.056 / +0.042
[16:03:15] hqnn_noent_noreup_q4_l2_s1: training
[16:04:35] hqnn_noent_noreup_q4_l2_s1: params 287 (middle 24) | train 66 s | best epoch 15/18 | val MAE 0.2609 / 0.4072 / 0.5812 kW | skill +0.089 / +0.053 / +0.039
[16:04:35] ctrl_mlp_q4_l2_s1: training
[16:04:55] ctrl_mlp_q4_l2_s1: params 285 (middle 22) | train 4 s | best epoch 10/13 | val MAE 0.2657 / 0.4078 / 0.5812 kW | skill +0.073 / +0.051 / +0.039
[16:04:55] ct

In [6]:
# Cell 6: table, mean +- sd over seeds (validation), merged with the frozen Week 4 benchmark
rows = []
for name in NAMES:
    for seed in SEEDS:
        p = ROOT / "experiments" / GROUP / f"{name}_s{seed}" / "metrics.json"
        if not p.exists():
            print("missing:", p.parent.name)
            continue
        r = json.loads(p.read_text())
        for v in r["val"]:
            rows.append({"model": name, "seed": seed, **{k: v.get(k) for k in
                         ["h", "MAE", "RMSE", "sMAPE", "R2_delta", "skill_MAE", "MAE_ramp", "MAE_normal", "ramp_F1",
                          "MAE_no_mod0_target", "DM_p"]},
                         "n_params": r["info"]["n_params"], "n_params_middle": r["info"]["n_params_middle"],
                         "train_s": r["info"]["train_s"], "best_epoch": r["info"]["best_epoch"],
                         "cap_hit": r["info"].get("cap_hit", False),
                         "s_per_epoch": r["metrics"].get("mean_epoch_train_s"),
                         "infer_ms_per_sample": r["metrics"]["infer_ms_per_sample"]})
runs = pl.DataFrame(rows)
runs.write_csv(REP / "qnn_val_runs.csv")
summ = (runs.group_by("model", "h").agg(
            pl.len().alias("n_seeds"), pl.col("MAE").mean().alias("MAE_mean"), pl.col("MAE").std().alias("MAE_sd"),
            *[pl.col(c).mean() for c in ["RMSE", "sMAPE", "R2_delta", "skill_MAE", "MAE_ramp", "MAE_normal", "ramp_F1",
                                         "MAE_no_mod0_target"]],
            pl.col("skill_MAE").std().alias("skill_sd"), pl.col("DM_p").max().alias("DM_p"),
            pl.col("n_params").first(), pl.col("n_params_middle").first(), pl.col("train_s").mean(),
            pl.col("s_per_epoch").mean(), pl.col("best_epoch").mean(), pl.col("cap_hit").any(),
            pl.col("infer_ms_per_sample").mean())
          .with_columns(pl.when(pl.col("model").str.starts_with("hqnn")).then(pl.lit("hybrid quantum"))
                          .otherwise(pl.lit("classical control")).alias("family"))
          .sort("h", "MAE_mean"))
summ.write_csv(REP / "qnn_val_summary.csv")
pl.Config.set_tbl_rows(80); pl.Config.set_tbl_cols(20); pl.Config.set_tbl_width_chars(250)
print(summ.select("h", "model", "n_seeds", pl.col("MAE_mean").round(4), pl.col("MAE_sd").round(4),
                  pl.col("skill_MAE").round(3), pl.col("skill_sd").round(3), pl.col("MAE_ramp").round(3),
                  pl.col("MAE_normal").round(4), pl.col("ramp_F1").round(3), "n_params", "n_params_middle",
                  pl.col("s_per_epoch").round(1), pl.col("best_epoch").round(1), "cap_hit"))

w4 = pl.read_csv(ROOT / "reports" / "week4" / "benchmark_val_week4_all.csv")
new = (summ.with_columns(pl.lit(False).alias("covariates"))
           .select([c for c in w4.columns if c in summ.columns or c == "covariates"]))
bench = pl.concat([w4, new.with_columns([pl.col(c).cast(w4.schema[c]) for c in new.columns])], how="diagonal_relaxed")
bench = bench.sort("h", "MAE_mean").with_columns(pl.col("MAE_mean").rank("ordinal").over("h").alias("rank"))
bench.write_csv(REP / "benchmark_val_week5_qnn.csv")
for h in R.HORIZONS:
    print(f"\n=== h = {h} min: validation ranking ({bench.filter(pl.col('h') == h).height} models) ===")
    print(bench.filter((pl.col("h") == h) & (pl.col("model").is_in(NAMES + ["lightgbm", "patchtst", "gru_uni",
                       "dlinear", "nlinear", "ar_lags", "persistence"])))
               .select("rank", "family", "model", "n_seeds", pl.col("MAE_mean").round(4), pl.col("MAE_sd").round(4),
                       pl.col("skill_MAE").round(3)))
SUMMARY["val_summary"] = summ.to_dicts()

shape: (12, 15)
┌─────┬─────────────────────────┬─────────┬──────────┬────────┬───────────┬──────────┬──────────┬────────────┬─────────┬──────────┬─────────────────┬─────────────┬────────────┬─────────┐
│ h   ┆ model                   ┆ n_seeds ┆ MAE_mean ┆ MAE_sd ┆ skill_MAE ┆ skill_sd ┆ MAE_ramp ┆ MAE_normal ┆ ramp_F1 ┆ n_params ┆ n_params_middle ┆ s_per_epoch ┆ best_epoch ┆ cap_hit │
│ --- ┆ ---                     ┆ ---     ┆ ---      ┆ ---    ┆ ---       ┆ ---      ┆ ---      ┆ ---        ┆ ---     ┆ ---      ┆ ---             ┆ ---         ┆ ---        ┆ ---     │
│ i64 ┆ str                     ┆ u32     ┆ f64      ┆ f64    ┆ f64       ┆ f64      ┆ f64      ┆ f64        ┆ f64     ┆ i64      ┆ i64             ┆ f64         ┆ f64        ┆ bool    │
╞═════╪═════════════════════════╪═════════╪══════════╪════════╪═══════════╪══════════╪══════════╪════════════╪═════════╪══════════╪═════════════════╪═════════════╪════════════╪═════════╡
│ 1   ┆ hqnn_noent_noreup_q4_l2 ┆ 3       ┆ 0.261

In [7]:
# Cell 7: Diebold-Mariano (absolute errors, Newey-West lag 60, Holm per horizon) and ramp breakdown, seed 0
pred = C.load_predictions(ROOT, seed=0, log=lambda s: None)
parts = [pred]
for name in NAMES:
    p = ROOT / "experiments" / GROUP / f"{name}_s0" / "predictions_val.parquet"
    parts.append(C._std(pl.read_parquet(p), name))
pred = pl.concat(parts)
REFS = [m for m in ["patchtst", "gru_uni", "persistence"] if m in pred["model"].unique().to_list()]
print("references found:", REFS, "| missing:", sorted({"patchtst", "gru_uni", "persistence"} - set(REFS)))
MODELS = NAMES + REFS
al, cov = C.align(pred.filter(pl.col("model").is_in(MODELS)))
print(cov.group_by("h").agg(pl.col("n").min().alias("n_min"), pl.col("n_common").first()).sort("h"))
dm = C.dm_pairs(al, {h: MODELS for h in R.HORIZONS})
dm.write_csv(REP / "qnn_dm_pairs_val.csv")
main = NAMES[0]
print(dm.filter((pl.col("model_a") == main) | (pl.col("model_b") == main))
        .select("h", "model_a", "model_b", pl.col("MAE_a").round(4), pl.col("MAE_b").round(4),
                pl.col("DM_stat").round(2), "DM_p_holm", "better", "significant_5pct"))
lab = C.add_labels(al, md)
rb = C.ramp_breakdown(lab, meta["thresholds"])
rb.write_csv(REP / "qnn_ramp_breakdown_val.csv")
print(rb.select("h", "model", "n_ramp", pl.col("MAE_normal").round(4), pl.col("MAE_ramp").round(3),
                pl.col("skill_normal").round(3), pl.col("skill_ramp").round(3), pl.col("ramp_F1").round(3)))
SUMMARY["dm_main"] = dm.filter((pl.col("model_a") == main) | (pl.col("model_b") == main)).to_dicts()

references found: ['patchtst', 'gru_uni', 'persistence'] | missing: []
shape: (3, 3)
┌─────┬───────┬──────────┐
│ h   ┆ n_min ┆ n_common │
│ --- ┆ ---   ┆ ---      │
│ i64 ┆ u32   ┆ u32      │
╞═════╪═══════╪══════════╡
│ 1   ┆ 42575 ┆ 42575    │
│ 5   ┆ 42567 ┆ 42567    │
│ 15  ┆ 42547 ┆ 42547    │
└─────┴───────┴──────────┘
shape: (18, 9)
┌─────┬───────────────────┬─────────────────────────┬────────┬────────┬─────────┬────────────┬─────────────────────────┬──────────────────┐
│ h   ┆ model_a           ┆ model_b                 ┆ MAE_a  ┆ MAE_b  ┆ DM_stat ┆ DM_p_holm  ┆ better                  ┆ significant_5pct │
│ --- ┆ ---               ┆ ---                     ┆ ---    ┆ ---    ┆ ---     ┆ ---        ┆ ---                     ┆ ---              │
│ i64 ┆ str               ┆ str                     ┆ f64    ┆ f64    ┆ f64     ┆ f64        ┆ str                     ┆ bool             │
╞═════╪═══════════════════╪═════════════════════════╪════════╪════════╪═════════╪════════════╪═══

In [8]:
# Cell 8: summary
(REP / "qnn_val.json").write_text(json.dumps(SUMMARY, indent=1, default=str))
print("written:", REP / "qnn_val.json")
for h in R.HORIZONS:
    s = summ.filter(pl.col("h") == h)
    print(f"h={h:>2}: " + " | ".join(f"{m} {s.filter(pl.col('model') == m)['skill_MAE'][0]:+.3f}"
                                      for m in NAMES if s.filter(pl.col('model') == m).height))

written: /Users/aryaneuse/Desktop/Uni Project/datacenter-power-forecasting/reports/week5/qnn_val.json
h= 1: hqnn_noreup_q4_l2 +0.074 | hqnn_noent_noreup_q4_l2 +0.086 | ctrl_mlp_q4_l2 +0.056 | ctrl_bottleneck_q4_l2 +0.066
h= 5: hqnn_noreup_q4_l2 +0.055 | hqnn_noent_noreup_q4_l2 +0.054 | ctrl_mlp_q4_l2 +0.053 | ctrl_bottleneck_q4_l2 +0.045
h=15: hqnn_noreup_q4_l2 +0.043 | hqnn_noent_noreup_q4_l2 +0.041 | ctrl_mlp_q4_l2 +0.041 | ctrl_bottleneck_q4_l2 +0.033


In [9]:
# Cell 9: export outputs (save the notebook first with Cmd+S, then run this cell)
%run ../src/tools/nb_outputs_to_md.py 18_week5_qnn_val.ipynb

written 18_week5_qnn_val_outputs.md | code cells 9 | figures 0
